# Molecular representation learning 5-fold results demo

This notebook provides lightweight result tables for the RMMol GitHub repository. It demonstrates how to inspect 5-fold MolecularNet-style evaluation results, including BBBP, BACE, HIV, the FDA-approved ClinTox label, Tox21, SIDER, ESOL, FreeSolv, Lipophilicity and QM9.

Classification metrics are AUROC in percent for the cross-model source tables. Regression metrics are RMSE for ESOL/FreeSolv/Lipophilicity and MAE for QM9.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('data')
domain = pd.read_csv(DATA_DIR / 'moleculenet_four_domain_summary.csv')
task_source = pd.read_csv(DATA_DIR / 'moleculenet_task_source.csv')
confseq_summary = pd.read_csv(DATA_DIR / 'confseq_moleculenet_5fold_summary.csv')
confseq_folds = pd.read_csv(DATA_DIR / 'confseq_moleculenet_5fold_fold_metrics.csv')
bbbp_summary = pd.read_csv(DATA_DIR / 'bbbp_multimodel_5fold_summary.csv')
bbbp_folds = pd.read_csv(DATA_DIR / 'bbbp_multimodel_5fold_fold_metrics.csv')

domain

In [ ]:
# ConfSeq 5-fold benchmark summary. ClinTox is shown task-wise; use FDA_APPROVED for the paper-specific FDA-approved label.
cols = ['dataset', 'task', 'model', 'metric', 'mean', 'std', 'n_folds', 'probe']
confseq_summary[cols].query("task == '__aggregate__' or task == 'FDA_APPROVED' or dataset in ['ESOL', 'FreeSolv', 'Lipophilicity']").reset_index(drop=True)

In [ ]:
# BBBP fold-level comparison across models. Values are fold means from the packaged probe results.
bbbp_summary.sort_values('roc_auc_mean', ascending=False)

In [ ]:
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'font.size': 8,
    'figure.dpi': 160,
})
palette = {
    'RMMol': '#4C78A8', 'ConfSeq': '#72B7B2', 'MoLFormer': '#F58518',
    'MoleBERT': '#54A24B', 'GROVER': '#B279A2', 'UniMol': '#E45756', 'FCFP': '#8C8C8C',
}

fig, ax = plt.subplots(figsize=(7.2, 3.2))
sns.barplot(data=domain, x='domain', y='domain_score', hue='method', palette=palette, ax=ax, errorbar=None)
ax.set_xlabel('')
ax.set_ylabel('Domain score')
ax.tick_params(axis='x', rotation=20)
ax.legend(frameon=False, ncol=4, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, 1.28))
fig.tight_layout()

In [ ]:
# BBBP 5-fold distribution for each representation.
plot_df = bbbp_folds.copy()
plot_df['model_label'] = plot_df['model'].replace({'rmmol_true': 'RMMol', 'confseq': 'ConfSeq', 'molformer': 'MoLFormer', 'molebert': 'MoleBERT', 'grover': 'GROVER', 'unimol': 'UniMol', 'fcfp': 'FCFP'})
order = bbbp_summary.assign(model_label=lambda d: d['model'].replace({'rmmol_true': 'RMMol', 'confseq': 'ConfSeq', 'molformer': 'MoLFormer', 'molebert': 'MoleBERT', 'grover': 'GROVER', 'unimol': 'UniMol', 'fcfp': 'FCFP'})).sort_values('roc_auc_mean', ascending=False)['model_label']

fig, ax = plt.subplots(figsize=(7.4, 3.1))
sns.boxplot(data=plot_df, x='model_label', y='roc_auc', order=order, color='white', width=0.52, fliersize=0, ax=ax)
sns.stripplot(data=plot_df, x='model_label', y='roc_auc', order=order, hue='model_label', palette=palette, size=3.6, alpha=0.75, ax=ax, legend=False)
ax.set_xlabel('')
ax.set_ylabel('BBBP ROC-AUC')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()

In [ ]:
# Dataset-level source values used by the GitHub demo figures.
task_source.head(20)

In [ ]:
# Optional: uncomment to save a local figure.
# out_dir = Path('outputs')
# out_dir.mkdir(exist_ok=True)
# fig.savefig(out_dir / 'bbbp_5fold_demo.png', dpi=300, bbox_inches='tight')